# Hachimi Converter - Colab 训练

在 Colab GPU 上训练 Hachimi 风格转换 U-Net。

**使用前：**
1. 运行时 → 更改运行时类型 → GPU (T4)
2. 按顺序执行下面的 cell

## 0. 检查 GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("⚠️ 未检测到 GPU，请检查运行时设置")

## 1. 克隆仓库 & 安装依赖

In [ ]:
# TODO: 替换为你的仓库地址
REPO_URL = "https://github.com/YOUR_USER/hachimi-converter.git"

!git clone {REPO_URL} /content/hachimi-converter
%cd /content/hachimi-converter

In [ ]:
!pip install -q librosa soundfile matplotlib yt-dlp gdown

## 2. 下载训练数据

直接在 Colab 上从 Hachimi API + YouTube 下载，不需要代理。

In [ ]:
# Colab 不需要代理，直接运行即可
# 如果需要代理，加 --proxy http://your-proxy:port
%cd /content/hachimi-converter/scripts
!python download_v2.py

## 3. 验证配对质量

In [ ]:
!python validate_pairs.py

## 4. 对齐 & 切片

In [ ]:
!python slice_v2.py

In [ ]:
# 检查训练数据量
import glob
paired = glob.glob("/content/hachimi-converter/data/paired/*_hach_*.wav")
print(f"训练片段数: {len(paired)}")

## 5. 下载 HiFi-GAN 声码器权重

In [ ]:
!python download_vocoder.py

## 6. 训练

In [ ]:
!python train.py --device cuda --epochs 120 --batch-size 32

## 7. 试听效果

从已下载的原曲中随机选几首，跑推理后直接播放对比。

In [ ]:
import random
from pathlib import Path
from IPython.display import Audio, display, HTML

original_dir = Path("/content/hachimi-converter/data/original")
output_dir = Path("/content/hachimi-converter/output/demo")
output_dir.mkdir(parents=True, exist_ok=True)

# 随机选 3 首原曲（取 15 秒片段）
songs = list(original_dir.glob("*.wav"))
samples = random.sample(songs, min(3, len(songs)))

for song in samples:
    out_path = output_dir / f"{song.stem}_hachimi.wav"
    print(f"\n{'='*60}")
    print(f"转换: {song.stem}")
    !python inference.py "{song}" "{out_path}" 15 30

    display(HTML(f"<h4>{song.stem}</h4>"))
    display(HTML("<b>原曲 (30-45s):</b>"))
    display(Audio(str(song), rate=22050, normalize=False))
    display(HTML("<b>Hachimi 风格:</b>"))
    display(Audio(str(out_path), rate=22050, normalize=False))

## 8. 下载模型权重 & demo 音频

In [ ]:
from google.colab import files

# 下载模型
files.download("/content/hachimi-converter/models/hachimi_unet_best.pt")
files.download("/content/hachimi-converter/models/hachimi_unet_final.pt")

# 下载 demo 音频
for f in Path("/content/hachimi-converter/output/demo").glob("*.wav"):
    files.download(str(f))

## 9. (可选) 上传自己的音频测试

In [ ]:
from google.colab import files
uploaded = files.upload()
input_file = list(uploaded.keys())[0]
print(f"已上传: {input_file}")

!python inference.py "/content/{input_file}" /content/output.wav

from IPython.display import Audio
Audio("/content/output.wav")

In [ ]:
files.download("/content/output.wav")